In [2]:
import pandas as pd
import numpy as np
import requests

In [3]:
df = pd.read_json('../ex02/auto.json')
df.head(2)

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.0,Ford,Focus
1,E432XX77RUS,1,6500.0,Toyota,Camry


In [4]:
pd.options.display.float_format = '{:,.2f}'.format
df.head(2)

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,"3,200.00",Ford,Focus
1,E432XX77RUS,1,"6,500.00",Toyota,Camry


In [5]:
df.isna().sum()

CarNumber    0
Refund       0
Fines        0
Make         0
Model        9
dtype: int64

In [6]:
car_info = df[['CarNumber', 'Make', 'Model']].drop_duplicates()

sampled_cars = car_info.sample(n=200, replace=True, random_state=21)

sampled_fines = df['Fines'].sample(n=200, replace=True, random_state=21).reset_index(drop=True)
sampled_refunds = df['Refund'].sample(n=200, replace=True, random_state=21).reset_index(drop=True)

new_sample = sampled_cars.reset_index(drop=True).copy()
new_sample['Fines'] = sampled_fines
new_sample['Refund'] = sampled_refunds

concat_rows = pd.concat([df, new_sample], ignore_index=True)

print(df.shape, concat_rows.shape)

(725, 5) (925, 5)


In [7]:
np.random.seed(21)
years = pd.Series(np.random.randint(1980, 2020, size=concat_rows.shape[0]), name='Year')
fines = pd.concat([concat_rows.reset_index(drop=True), years], axis=1)
fines.head(2)

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,"3,200.00",Ford,Focus,1989
1,E432XX77RUS,1,"6,500.00",Toyota,Camry,1995


In [8]:
import json
import re

with open('../../datasets/surname.json', 'r') as f:
    surnames_raw = json.load(f)

if isinstance(surnames_raw[0], dict) and 'name' in surnames_raw[0]:
    raw_names = [item['name'] for item in surnames_raw if 'name' in item]
else:
    raw_names = [str(item) for item in surnames_raw]

clean_surnames = [re.sub(r'[^\w\s]', '', name).strip() for name in raw_names]
unique_cars = df['CarNumber'].drop_duplicates().reset_index(drop=True)

np.random.seed(21)
sampled_surnames = pd.Series(
    np.random.choice(clean_surnames, size=len(unique_cars), replace=True),
    name='SURNAME'
)

owners = pd.DataFrame({'CarNumber': unique_cars, 'SURNAME': sampled_surnames})
owners.head(2)

,CarNumber,SURNAME
0,Y163O8161RUS,REYES 327904 60
1,E432XX77RUS,ROGERS 302261 66


In [9]:
new_fines = pd.DataFrame({
    'CarNumber' : ['ABC123', 'XYZ789', 'QWE456', 'LMN321', 'GHJ678'],
    'Fine' : [120, 85, 150, 200, 95],
    'Year' : [2023, 2022, 2024, 2023, 2022]
})

fines = pd.concat([fines, new_fines], ignore_index=True)

In [10]:
fines.sample(2)

,CarNumber,Refund,Fines,Make,Model,Year,Fine
303,C313MM99RUS,1.00,"7,500.00",Toyota,Camry,2019,NaN
797,Y359O8197RUS,1.00,"8,500.00",Ford,Focus,1987,NaN


In [11]:
owners = owners[:-20]

new_owners = pd.DataFrame({
    'CarNumber' : ['A111AA777RUS', 'A222AA777RUS', 'A333AA777RUS'],
    'SURNAME' : ['PETROV 111111 1', 'IVANOV 222222 2', 'SIDOROV 333333 3']
})

owners = pd.concat([owners, new_owners], ignore_index=True)

In [12]:
owners.sample(5)

,CarNumber,SURNAME
246,X522OM161RUS,GARCIA 1166120 6
297,X255HE161RUS,KIM 262352 77
67,O719MM163RUS,WATSON 252579 81
266,T6559O50RUS,ROBINSON 529821 30
64,T6449O50RUS,BROOKS 251663 82


In [13]:
inner_df = pd.merge(fines, owners, on='CarNumber', how='inner')
outer_df = pd.merge(fines, owners, on='CarNumber', how='outer')
left_df = pd.merge(fines, owners, on='CarNumber', how='left')
right_df = pd.merge(fines, owners, on='CarNumber', how='right')

In [14]:
pivot_table = fines.pivot_table(index=['Make', 'Model'], columns='Year', values='Fines', aggfunc='sum')

In [15]:
pivot_table

Year                    1980       1981       1982      1983       1984  \
Make       Model                                                          
Ford       Focus   56,489.17 398,589.17 140,383.76 62,300.00 112,494.59   
           Mondeo        NaN        NaN        NaN       NaN        NaN   
Skoda      Octavia  1,900.00        NaN   6,900.00 11,594.59        NaN   
Toyota     Camry   28,500.00   8,594.59        NaN  7,200.00        NaN   
           Corolla       NaN        NaN   2,000.00    800.00        NaN   
Volkswagen Golf    30,900.00        NaN        NaN  8,594.59     300.00   
           Jetta         NaN        NaN        NaN       NaN        NaN   
           Passat   6,500.00   1,600.00        NaN  3,200.00  10,000.00   
           Touareg       NaN        NaN        NaN       NaN        NaN   

Year                     1985       1986       1987      1988       1989  ...  \
Make       Model                                                          ...   
Ford       Focus   189,583.76 104,994.59 132,800.00 95,489.17 125,700.00  ...   
           Mondeo         NaN        NaN        NaN       NaN   8,600.00  ...   
Skoda      Octavia  10,294.59     600.00   5,200.00    500.00  91,400.00  ...   
Toyota     Camry          NaN        NaN        NaN       NaN  22,400.00  ...   
           Corolla        NaN        NaN   8,000.00       NaN   4,000.00  ...   
Volkswagen Golf     24,000.00        NaN   9,300.00       NaN   5,800.00  ...   
           Jetta          NaN        NaN        NaN       NaN        NaN  ...   
           Passat    5,000.00  15,000.00  12,300.00       NaN        NaN  ...   
           Touareg   5,800.00        NaN        NaN       NaN        NaN  ...   

Year                     2010      2011       2012       2013       2014  \
Make       Model                                                           
Ford       Focus   120,183.76 86,689.17 120,200.00 149,294.59 157,494.59   
           Mondeo         NaN       NaN  34,400.00        NaN        NaN   
Skoda      Octavia   3,100.00    500.00     500.00  19,594.59   3,300.00   
Toyota     Camry          NaN  3,300.00  10,594.59        NaN        NaN   
           Corolla  24,000.00  8,594.59        NaN        NaN        NaN   
Volkswagen Golf           NaN    300.00        NaN        NaN        NaN   
           Jetta          NaN       NaN        NaN        NaN        NaN   
           Passat    2,800.00       NaN        NaN        NaN        NaN   
           Touareg   6,300.00       NaN        NaN        NaN   1,300.00   

Year                     2015      2016       2017       2018       2019  
Make       Model                                                          
Ford       Focus   210,789.17 83,694.59 268,200.00 283,594.59 117,100.00  
           Mondeo         NaN 48,100.00        NaN        NaN        NaN  
Skoda      Octavia  46,394.59    300.00   4,000.00 156,200.00   9,500.00  
Toyota     Camry          NaN       NaN   1,000.00  13,000.00  18,100.00  
           Corolla        NaN       NaN   9,600.00        NaN        NaN  
Volkswagen Golf      2,300.00       NaN        NaN        NaN        NaN  
           Jetta          NaN       NaN        NaN        NaN        NaN  
           Passat      600.00  2,100.00        NaN        NaN        NaN  
           Touareg     500.00       NaN        NaN        NaN        NaN  

[9 rows x 40 columns]

In [17]:
fines.to_csv('fines.csv', index=False)
owners.to_csv('owners.csv', index=False)